In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 모달리티 비교: FP(fingerprint) / IMG(OCSR image) / SMI(ChemBERTa) / GNN(MGSSL) /
# Conf(ConforFormer 3D conformation) 각각을 단독으로 DownstreamMLP에 넣었을 때의 성능과,
# CUE(=CUE, softmax fusion으로 FP+IMG를 결합한 기존 파이프라인)를 비교한다.
# SMI/GNN/Conf는 cue4modal 저장소(cue4modal/smi_train.py, gnn_mgssl_train.py, conf_train.py)에서
# 임베딩을 캐싱한 뒤 DownstreamMLP만 새로 학습한 결과이며, FP/IMG/CUE(cmfe)와 동일한
# 5-fold 내부 CV·metrics.csv 스키마(train.py::aggregate_metrics)를 공유한다.
cue_base = '/data/rlawlsgurjh/work/gitlab/CUE/cue/results'
cue4modal_base = '/data/rlawlsgurjh/work/gitlab/CUE/cue4modal/results'

datasets = ['DIS', 'INH']
targets = ['PI', 'Ssel', 'WS2', 'RS2']
metrics_of_interest = ['rmse', 'r2', 'pcc']

# 비교할 소스: (표시 이름, metrics.csv 루트 경로). CUE는 항상 마지막에 오도록 순서를 고정한다.
# FP/IMG/CUE는 cue_base, SMI/GNN/Conf는 cue4modal_base에서 가져온다.
model_sources = {
    'FP': os.path.join(cue_base, '{dataset}', '{target}', 'v1', 'FP'),
    'IMG': os.path.join(cue_base, '{dataset}', '{target}', 'v1', 'IMG'),
    'SMI': os.path.join(cue4modal_base, '{dataset}', '{target}', 'v1', 'SMI'),
    'GNN': os.path.join(cue4modal_base, '{dataset}', '{target}', 'v2', 'GNN_MGSSL'),
    'Conf': os.path.join(cue4modal_base, '{dataset}', '{target}', 'v1', 'CONF'),
    'CUE': os.path.join(cue_base, '{dataset}', '{target}', 'v1', 'CUE'),
}

model_order = ['FP', 'IMG', 'SMI', 'GNN', 'Conf', 'CUE']

# rmse는 낮을수록, r2/pcc는 높을수록 좋은 성능
LOWER_IS_BETTER = {'rmse': True, 'r2': False, 'pcc': False}

In [2]:
def load_metric(model_dir_template, dataset, target):
    """cmfe의 metrics.csv(train.py::aggregate_metrics 출력)를 읽어
    model_name == 'mlp' 행의 rmse/r2/pcc(+ _std)를 뽑아온다.
    metrics.csv가 아직 없으면(해당 모달리티를 아직 안 돌렸으면) None을 반환한다."""
    metrics_path = os.path.join(model_dir_template.format(dataset=dataset, target=target), 'metrics.csv')
    if not os.path.exists(metrics_path):
        return None

    df_m = pd.read_csv(metrics_path)
    mlp_row = df_m[df_m['model_name'] == 'mlp']
    if mlp_row.empty:
        return None

    values = {}
    for col in mlp_row.columns:
        if col in ['model_name', 'Unnamed: 0']:
            continue
        norm_col = col.lower()
        if norm_col in metrics_of_interest or norm_col.replace('_std', '') in metrics_of_interest:
            values[norm_col] = mlp_row[col].values[0]
    return values if values else None


raw_rows = []
for d_name in datasets:
    for target in targets:
        for model_name, dir_template in model_sources.items():
            vals = load_metric(dir_template, d_name, target)
            if vals is not None:
                raw_rows.append({'Dataset': d_name, 'Target': target, 'Model': model_name, **vals})

df_total = pd.DataFrame(raw_rows)
n_expected = len(datasets) * len(targets) * len(model_sources)
print(f"✅ 데이터 로드 완료: {len(df_total)}개의 항목이 수집되었습니다 (기대값: {len(datasets)} x {len(targets)} x {len(model_sources)} = {n_expected}).")
if len(df_total) < n_expected:
    missing = n_expected - len(df_total)
    have = sorted(df_total['Model'].unique().tolist()) if not df_total.empty else []
    print(f"⚠️  아직 {missing}개 항목이 없습니다. 현재 확보된 모델: {have}")
    print("   (smi_train.py / gnn_train.py / conf_train.py 를 돌려 metrics.csv를 생성한 뒤 이 셀부터 다시 실행하면 됩니다.)")

✅ 데이터 로드 완료: 48개의 항목이 수집되었습니다 (기대값: 2 x 4 x 6 = 48).


In [3]:
def plot_comparison_styled(df_plot, dataset, target_name, metric_name, output_dir):
    """지표 라벨은 지우되, Y축 눈금 수치와 배경은 유지한 버전.
    FP/IMG/SMI/GNN/Conf/CUE를 해치 없이 colors 색상으로만 구분한다 (CUE가 항상 마지막 막대).
    막대 높이도 텍스트 라벨과 동일하게 소수점 셋째 자리까지 반올림한 값을 사용한다."""
    df_target = df_plot.set_index('Model').reindex(model_order).dropna(subset=[metric_name]).reset_index()
    if df_target.empty: return

    models = df_target['Model'].tolist()
    values = [round(v, 3) for v in df_target[metric_name].tolist()]
    # model_order 순서에 맞춰 6색을 그대로 사용한다 (일부 모달리티가 아직 없으면 앞에서부터 사용).
    colors = ['#ef476f', '#f78c6b', '#ffd166', '#06d6a0', '#118ab2', '#073b4c']

    y_min, y_max = min(values), max(values)
    margin = (y_max - y_min) * 0.45 if (y_max - y_min) != 0 else 0.1

    fig, ax = plt.subplots(figsize=(8, 4))
    x = np.arange(len(models))

    bars = ax.bar(x, values, color=colors[:len(models)], edgecolor='black', alpha=0.9, linewidth=1)

    for i, bar in enumerate(bars):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + (margin * 0.03),
                f'{height:.3f}', ha='center', va='bottom', fontsize=13,
                fontweight='bold')

    ax.set_title("")
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_xticklabels([])

    ax.tick_params(axis='y', labelsize=11)
    ax.tick_params(axis='x', which='both', length=0)

    ax.grid(axis='y', linestyle='--', alpha=0.5)
    for spine in ax.spines.values():
        spine.set_visible(True)

    ax.set_ylim(max(0, y_min - (margin*0.1)), y_max + margin)

    plt.tight_layout()
    os.makedirs(output_dir, exist_ok=True)
    save_path = os.path.join(output_dir, f"{target_name}_{metric_name}.png")
    plt.savefig(save_path, dpi=300)
    plt.close()
    print(f"✅ 저장 완료: {save_path}")

In [4]:
plot_base_dir = os.path.join(cue_base, 'plots', 'modal_comparison_v2')

for d_name in datasets:
    for target in targets:
        df_target = df_total[(df_total['Dataset'] == d_name) & (df_total['Target'] == target)]
        if df_target.empty: continue

        g_name = 'DIS' if d_name == 'DIS' else 'INH'
        output_path = os.path.join(plot_base_dir, g_name)
        for m in metrics_of_interest:
            if m in df_target.columns:
                plot_comparison_styled(df_target, d_name, target, m, output_path)

✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/modal_comparison_v2/DIS/PI_rmse.png
✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/modal_comparison_v2/DIS/PI_r2.png


✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/modal_comparison_v2/DIS/PI_pcc.png
✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/modal_comparison_v2/DIS/Ssel_rmse.png


✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/modal_comparison_v2/DIS/Ssel_r2.png


✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/modal_comparison_v2/DIS/Ssel_pcc.png
✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/modal_comparison_v2/DIS/WS(alpha=2)_rmse.png


✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/modal_comparison_v2/DIS/WS(alpha=2)_r2.png
✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/modal_comparison_v2/DIS/WS(alpha=2)_pcc.png


✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/modal_comparison_v2/DIS/RS(k=2)_rmse.png
✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/modal_comparison_v2/DIS/RS(k=2)_r2.png


... (5 outputs left out)

✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/modal_comparison_v2/INH/Ssel_pcc.png
✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/modal_comparison_v2/INH/WS(alpha=2)_rmse.png


✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/modal_comparison_v2/INH/WS(alpha=2)_r2.png
✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/modal_comparison_v2/INH/WS(alpha=2)_pcc.png


✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/modal_comparison_v2/INH/RS(k=2)_rmse.png
✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/modal_comparison_v2/INH/RS(k=2)_r2.png


✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/modal_comparison_v2/INH/RS(k=2)_pcc.png
